In [33]:
import numpy as np
import pandas as pd
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
import warnings
warnings.simplefilter("ignore")

In [34]:
path = "/Users/aryagupta/Desktop/food-delivery-estimation/food_time_prediction_using_mlops/data/raw/train.csv"

data = pd.read_csv(path)
data = data.drop([45593], axis = 0) # contains all null value

In [35]:
pd.set_option('display.max_columns', 100)

In [36]:
data.columns = data.columns.str.lower()

data.rename({
        "delivery_person_id": "person_id", 
        "delivery_person_age" : "age", 
        "delivery_person_ratings" : "ratings",
        "delivery_location_latitude": "delivery_latitude",
        "delivery_location_longitude" : "delivery_longitude",
        "time_order_picked" : "order_picked",
        "weatherconditions" : "weather",
        "road_traffic_density" : "traffic", 
        "type_of_order" : "order_type",
        "time_taken(min)" : "time",
        "city" : 'city_category',
        'festival' : 'is_festival',
        'type_of_vehicle' : 'vehical_type',
        'time_orderd' : 'order_time',                        
    }, axis = 1, inplace = True)

In [37]:
data.columns

Index(['id', 'person_id', 'age', 'ratings', 'restaurant_latitude',
       'restaurant_longitude', 'delivery_latitude', 'delivery_longitude',
       'order_date', 'order_time', 'order_picked', 'weather', 'traffic',
       'vehicle_condition', 'order_type', 'vehical_type',
       'multiple_deliveries', 'is_festival', 'city_category', 'time'],
      dtype='object')

In [38]:
six_star_ratings = data[data['ratings'] == 6]
six_star_ratings.head()

,id,person_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,order_time,order_picked,weather,traffic,vehicle_condition,order_type,vehical_type,multiple_deliveries,is_festival,city_category,time


In [39]:
# don't delete
age_equal_15 = data[data['age'] == 15]
age_equal_15.head()

,id,person_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,order_time,order_picked,weather,traffic,vehicle_condition,order_type,vehical_type,multiple_deliveries,is_festival,city_category,time


In [42]:
def hello(data, age_equal_15, six_star_ratings):
    # # Lowercase column names in-place
    # data.columns = data.columns.str.lower()
    
    # # Rename columns in-place
    # data.rename({
    #     "delivery_person_id": "person_id", 
    #     "delivery_person_age" : "age", 
    #     "delivery_person_ratings" : "ratings",
    #     "delivery_location_latitude": "delivery_latitude",
    #     "delivery_location_longitude" : "delivery_longitude",
    #     "time_order_picked" : "order_picked",
    #     "weatherconditions" : "weather",
    #     "road_traffic_density" : "traffic", 
    #     "type_of_order" : "order_type",
    #     "time_taken(min)" : "time",
    #     "city" : 'city_category',
    #     'festival' : 'is_festival',
    #     'type_of_vehicle' : 'vehical_type',
    #     'time_orderd' : 'order_time',                        
    # }, axis=1, inplace=True)

    # ID Feature
    data['id'] = data['id'].replace("NaN", np.nan)

    # city colm
    data['city'] = data['person_id'].str.split('RES').str.get(0)

    # person_id
    data['person_id'] = data['person_id'].replace("NaN", np.nan)

    # age
    data['age'] = data['age'].replace("NaN", np.nan)
    data['age'] = data['age'].astype('float').round()
    
    # ratings
    data['ratings'] = data['ratings'].replace("NaN", np.nan)
    data['ratings'] = data['ratings'].astype('float')

    # restaurant_latitude
    data['restaurant_latitude'] = data['restaurant_latitude'].replace("NaN", np.nan)

    # restaurant_longitude
    data['restaurant_longitude'] = data['restaurant_longitude'].replace("NaN", np.nan)

    # delivery_latitude
    data['delivery_latitude'] = data['delivery_latitude'].replace("NaN", np.nan)

    # delivery_longitude
    data['delivery_longitude'] = data['delivery_longitude'].replace("NaN", np.nan)

    # order_date
    data['order_date'] = pd.to_datetime(data['order_date'], errors='coerce')

    # time_ordered
    data['order_time'] = data['order_time'].replace("NaN", np.nan)
    data['order_time'] = pd.to_datetime(data['order_time'], errors='coerce')

    # order_picked
    data['order_picked'] = pd.to_datetime(data['order_picked'], errors='coerce')

    # weather
    data['weather'] = data['weather'].replace("conditions NaN", np.nan)
    data['weather'] = data['weather'].str.lower()
    data['weather'] = data['weather'].str.replace("conditions ", "").str.strip()

    # traffic
    data['traffic'] = data['traffic'].replace("NaN ", np.nan)
    data['traffic'] = data['traffic'].str.lower()

    # vehical_condition
    data['vehicle_condition'] = data['vehicle_condition'].replace("NaN", np.nan)
    data['vehicle_condition'] = data['vehicle_condition'].astype('Int64')

    # order_type
    data['order_type'] = data['order_type'].replace("NaN ", np.nan)
    data['order_type'] = data['order_type'].str.lower()

    # type_of_vehicle
    data['vehical_type'] = data['vehical_type'].replace("NaN ", np.nan)

    # multiple_deliveries
    data['multiple_deliveries'] = data['multiple_deliveries'].replace("NaN ", np.nan)
    data['multiple_deliveries'] = data['multiple_deliveries'].astype('float')

    # festival
    data['is_festival'] = data['is_festival'].replace("NaN ", np.nan)
    data['is_festival'] = data['is_festival'].str.lower()

    # city
    data['city_category'] = data['city_category'].replace("NaN ", np.nan)
    data['city_category'] = data['city_category'].str.lower()

    # time
    data['time'] = data['time'].str.replace(r"\(min\)", "", regex=True)
    # data['time'] = pd.to_numeric(data['time'], errors='coerce')

    # Drop rows where age is 15
    data = data.drop(index = age_equal_15.index)
    data = data.drop(index = six_star_ratings.index)

    # Latitude/Longitude validation and cleaning
    loc_columns = ['restaurant_latitude', 'restaurant_longitude', 'delivery_latitude', 'delivery_longitude']
    lower_bound_lat_ind = 6.44
    lower_bound_long_ind = 68.70

    # Replace invalid lat/long values with NaN
    for col in loc_columns:
        if "latitude" in col:
            data[col] = np.where(data[col] < lower_bound_lat_ind, np.nan, data[col])
        elif "longitude" in col:
            data[col] = np.where(data[col] < lower_bound_long_ind, np.nan, data[col])

    # Datetime feature extraction
    date_col = pd.to_datetime(data['order_date'], dayfirst=True)
    data['day'] = date_col.dt.day
    data['month'] = date_col.dt.month
    data['year'] = date_col.dt.year
    data['day_of_week'] = date_col.dt.day_name()
    data['is_weekend'] = date_col.dt.day_name().isin(["Saturday", "Sunday"]).astype(int)

    # Hour and time of day from order time
    order_hour = pd.to_datetime(data['order_time'], errors='coerce').dt.hour
    data['order_time_hour'] = order_hour

    def time_of_day(ser: pd.Series):

        return(
            pd.cut(ser,bins=[0,6,12,17,20,24],right=True,
                   labels=["after_midnight","morning","afternoon","evening","night"])
        )

    data['order_time_of_day'] = time_of_day(order_hour)

    # Pickup time in minutes
    valid_times = data[['order_time', 'order_picked']].dropna()
    pickup_duration = (valid_times['order_picked'] - valid_times['order_time']).dt.total_seconds() / 60
    data['pickup_time'] = pickup_duration

    def calculate_haversine_distance(data, loc_columns):
        lat1 = data[loc_columns[0]]
        lon1 = data[loc_columns[1]]
        lat2 = data[loc_columns[2]]
        lon2 = data[loc_columns[3]]
        lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
        c = 2 * np.arcsin(np.sqrt(a))
        distance = 6371 * c  # Earth radius in km
        return data.assign(distance=distance)

    data = calculate_haversine_distance(data, loc_columns)

    def create_distance_type(data: pd.DataFrame) -> None:
        data["distance_type"] = pd.cut(
            data["distance"],
            bins=[0, 5, 10, 15, 25],
            right=False,
            labels=["short", "medium", "long", "very_long"]
        )

    create_distance_type(data)    

    data.to_csv("/Users/aryagupta/Desktop/food-delivery-estimation/food_time_prediction_using_mlops/data/interim/train_interim.csv")
    return data.isna().sum()


hello(data, age_equal_15, six_star_ratings)

id                         0
person_id                  0
age                     1854
ratings                 1908
restaurant_latitude     4071
restaurant_longitude    3802
delivery_latitude       3640
delivery_longitude      3640
order_date                 0
order_time              1731
order_picked               0
weather                  616
traffic                  601
vehicle_condition          0
order_type                 0
vehical_type               0
multiple_deliveries      993
is_festival              228
city_category           1200
time                       0
city                       0
day                        0
month                      0
year                       0
day_of_week                0
is_weekend                 0
order_time_hour         1731
order_time_of_day       2161
pickup_time             1731
distance                4071
distance_type           4071
dtype: int64

In [43]:
data.isna().sum()

id                         0
person_id                  0
age                     1854
ratings                 1908
restaurant_latitude        0
restaurant_longitude       0
delivery_latitude          0
delivery_longitude         0
order_date                 0
order_time              1731
order_picked               0
weather                  616
traffic                  601
vehicle_condition          0
order_type                 0
vehical_type               0
multiple_deliveries      993
is_festival              228
city_category           1200
time                       0
city                       0
dtype: int64